In [1]:
from migration.datasets import create_AIS_dataset

In [2]:
batch_size = 32

In [3]:
import tensorflow as tf
inputs, targets, mmsis, time_starts, time_ends, lengths, mean =  create_AIS_dataset('../../data/ct_2017010203_10_20/ct_2017010203_10_20_train.pkl', 
                   '../../data/ct_2017010203_10_20/mean.pkl',
                   batch_size,
                   99999, # not used lol
                   300,
                   300, 
                   30,
                   72, 
                   shuffle=False,
                   repeat=False)

batch_time_mask = tf.expand_dims(tf.transpose(tf.sequence_mask(lengths, dtype=inputs.dtype)), 2)


Instructions for updating:
tf.py_func is deprecated in TF V2. Instead, there are two
    options available in V2.
    - tf.py_function takes a python function which manipulates tf eager
    tensors instead of numpy arrays. It's easy to convert a tf eager tensor to
    an ndarray (just call tensor.numpy()) but having access to eager tensors
    means `tf.py_function`s can use accelerators such as GPUs as well as
    being differentiable using a gradient tape.
    - tf.numpy_function maintains the semantics of the deprecated tf.py_func
    (it is not differentiable, and manipulates numpy arrays). It drops the
    stateful argument making all functions stateful.
    
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `tf.cast` instead.
Instructions for updating:
Use `for ... in dataset:` to iterate over a dataset. If using `tf.estimator`, return the `Dataset` object directly from your input function. As a last resort, you can use `tf.compat.v1.data.make_one_

In [4]:
import tensorflow as tf
with tf.Session() as sess:
    inputs, targets, mmsis, time_starts, time_ends, lengths, mean, batch_time_mask = sess.run([inputs, targets, mmsis, time_starts, time_ends, lengths, mean, batch_time_mask])
    

2025-06-20 10:55:22.466027: I tensorflow/core/platform/cpu_feature_guard.cc:142] Your CPU supports instructions that this TensorFlow binary was not compiled to use: AVX2 FMA
2025-06-20 10:55:22.475623: I tensorflow/core/platform/profile_utils/cpu_utils.cc:94] CPU Frequency: 2688010000 Hz
2025-06-20 10:55:22.477314: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x3a68e8b0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
2025-06-20 10:55:22.477384: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): Host, Default Version


In [5]:
mean.shape

(702,)

In [6]:
print(inputs.shape)
assert (inputs.shape == targets.shape)

(99, 32, 702)


target is just input shifted one place and without mean substracted, both masked positions are cero

In [7]:
batch_time_mask.shape

(99, 32, 1)

In [62]:
mmsi_test_data = tf.constant(
    [
        [[1.0, 2.0], 
         [3.0, 4.0],
         [5.0, 6.0],
         [7.0, 8.0]],
         
        [[1.2, 2.2], 
         [3.3, 4.3],
         [0, 0],
         [0, 0]],
    ])

test_length = tf.convert_to_tensor([4,2])
test_mean = tf.convert_to_tensor([1,1.0])

def process_AIS_batch(data, lengths, mean):
    """Create mean-centered and time-major next-step prediction Tensors."""
    data = tf.to_float(tf.transpose(data, perm=[1, 0, 2]))
    mean = tf.to_float(mean)
    lengths = tf.to_int32(lengths)
    targets = data

    # Mean center the inputs.
    inputs = data - mean
    # Shift the inputs one step forward in time. Also remove the last
    # timestep so that targets and inputs are the same length.
    inputs = tf.pad(data, [[1, 0], [0, 0], [0, 0]], mode="CONSTANT")[:-1]
    # Mask out unused timesteps.
    inputs *= tf.expand_dims(tf.transpose(
        tf.sequence_mask(lengths, dtype=inputs.dtype)), 2)
    return inputs, targets

test_inputs, test_targets = process_AIS_batch(mmsi_test_data, test_length, test_mean)


In [64]:
with tf.Session() as sess:
    test_inputs, test_targets = sess.run([test_inputs, test_targets])

In [66]:
test_inputs

array([[[0. , 0. ],
        [0. , 0. ]],

       [[1. , 2. ],
        [1.2, 2.2]],

       [[3. , 4. ],
        [0. , 0. ]],

       [[5. , 6. ],
        [0. , 0. ]]], dtype=float32)

In [65]:
test_targets

array([[[1. , 2. ],
        [1.2, 2.2]],

       [[3. , 4. ],
        [3.3, 4.3]],

       [[5. , 6. ],
        [0. , 0. ]],

       [[7. , 8. ],
        [0. , 0. ]]], dtype=float32)

In [67]:
print(mmsis)
print(len(mmsis))

[304655000 304655000 236631000 236631000 236187000 236187000 311044200
 311044200 311044200 305555000 305555000 305808000 246061000 244366000
 244366000 244366000 244796000 244796000 244796000 538090070 538090070
 304407000 304407000 304407000 304407000 304407000 304407000 564919000
 636017516 636017516 250001109 250001109]
32


In [68]:
print(time_starts)
print(len(time_starts))

[1.4832288e+09 1.4891593e+09 1.4834349e+09 1.4875086e+09 1.4832288e+09
 1.4837780e+09 1.4840634e+09 1.4868408e+09 1.4876600e+09 1.4832288e+09
 1.4879027e+09 1.4832288e+09 1.4861757e+09 1.4832288e+09 1.4856737e+09
 1.4889797e+09 1.4832289e+09 1.4851105e+09 1.4891576e+09 1.4832289e+09
 1.4849838e+09 1.4832289e+09 1.4838957e+09 1.4845748e+09 1.4861736e+09
 1.4868836e+09 1.4885906e+09 1.4832289e+09 1.4832289e+09 1.4846417e+09
 1.4832291e+09 1.4837313e+09]
32


In [69]:
print(time_ends)
assert len(time_ends) == batch_size


[1.4832692e+09 1.4891867e+09 1.4834588e+09 1.4875341e+09 1.4832477e+09
 1.4837946e+09 1.4841032e+09 1.4868648e+09 1.4877042e+09 1.4832522e+09
 1.4879459e+09 1.4832543e+09 1.4862144e+09 1.4832520e+09 1.4857152e+09
 1.4890097e+09 1.4832442e+09 1.4851465e+09 1.4891864e+09 1.4832499e+09
 1.4850057e+09 1.4832486e+09 1.4839500e+09 1.4846336e+09 1.4862167e+09
 1.4869039e+09 1.4886222e+09 1.4832502e+09 1.4832557e+09 1.4846588e+09
 1.4832588e+09 1.4837731e+09]


In [70]:
assert(time_ends-time_starts > 0).all()

In [71]:
print(lengths)
assert (lengths <= inputs.shape[0]).all()
assert len(lengths) == batch_size

[68 46 41 43 32 28 67 41 74 40 73 43 65 39 70 51 26 61 49 36 37 34 91 99
 73 35 53 36 45 29 50 70]
